In [40]:
# Core imports
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Local imports
from src.config import config
from src.data_builder import DataBuilder
from src.econometrics import STRModelNoRestrictionsLSTR2

# Style
plt.rcParams.update({
    'figure.dpi': 300,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica'],
    'font.size': 11,
    'axes.edgecolor': 'black',
    'axes.facecolor': 'white',
    'axes.spines.top': True,
    'axes.spines.right': True,
    'axes.spines.left': True,
    'axes.spines.bottom': True,
    'axes.grid': False,
})

# Base output directory (we will create a mode-specific subfolder)
base_output_dir = config.OUTPUT_DIR
print('Secondary LSTR2 experiments notebook loaded.')

Secondary LSTR2 experiments notebook loaded.


In [42]:
"""
Dataset selection and output directories
"""
# Choose dataset
USE_MAX_DATASET = True  # set True to use maximum-length dataset

from pathlib import Path
import pandas as pd

# Reload data_builder module to pick up any changes
import importlib
import src.data_builder
importlib.reload(src.data_builder)
from src.data_builder import DataBuilder

max_path = config.DATA_DIR / 'df_panel_max.csv'
restricted_path = config.DATA_DIR / 'df_panel_final.csv'

def load_csv_safe(path: Path):
    if path.exists():
        df = pd.read_csv(path, parse_dates=['date'])
        if len(df) == 0:
            print(f"{path.name} is empty; ignoring.")
            return None
        return df
    return None

if USE_MAX_DATASET:
    df_panel = load_csv_safe(max_path)
    if df_panel is None:
        print('Max dataset not found. Building dataset from DBnomics...')
        builder = DataBuilder()
        df_panel = builder.run(build_mode='max')
        if len(df_panel) == 0:
            print('Max build returned empty dataset. Falling back to restricted.')
            df_panel = load_csv_safe(restricted_path)
            if df_panel is None:
                df_panel = builder.run(build_mode='restricted')
    else:
        print(f'Loaded existing max dataset from {max_path.name}')
else:
    df_panel = load_csv_safe(restricted_path)
    if df_panel is None:
        print('Building restricted dataset from DBnomics...')
        builder = DataBuilder()
        df_panel = builder.run(build_mode='restricted')
    else:
        print(f'Loaded existing restricted dataset from {restricted_path.name}')

# Countries dictionary (same mapping as main)
COUNTRIES = {
    'Australia': 'AUS',
    'Brazil': 'BRA',
    'Canada': 'CAN',
    'Euro Area': 'EA',
    'Indonesia': 'IND',
    'Japan': 'JPN',
    'Korea': 'KOR',
    'Mexico': 'MEX',
    'New Zealand': 'NZL',
    'Philippines': 'PHL',
    'Switzerland': 'CHE',
    'Thailand': 'THA',
    'Türkiye': 'TUR',
    'United Kingdom': 'GBR'
}

# Configure experiment-specific output directories
mode_folder = 'max' if USE_MAX_DATASET else 'restricted'
output_dir = base_output_dir / mode_folder / 'lstr2_unrestricted'
tables_dir = output_dir / 'tables'
figures_dir = output_dir / 'figures'
agg_fig_dir = figures_dir / 'aggregate'
country_fig_dir = figures_dir / 'country'
for d in [tables_dir, agg_fig_dir, country_fig_dir]:
    d.mkdir(parents=True, exist_ok=True)

print('Output directories:')
print(f'  Tables: {tables_dir}')
print(f'  Aggregate Figures: {agg_fig_dir}')
print(f'  Country Figures: {country_fig_dir}')

print('\nDataset Summary:')
print(f'  Total observations: {len(df_panel):,}')
print(f'  Countries: {df_panel["country"].nunique()}')
print(f"  Date range: {df_panel['date'].min()} to {df_panel['date'].max()}")
print('\nObservations per country:')
print(df_panel.groupby('country').agg(
    N=('date', 'count'),
    Start=('date', 'min'),
    End=('date', 'max')
).to_string())

Loaded existing max dataset from df_panel_max.csv
Output directories:
  Tables: /Users/lollo/Documents/Current_Projects/str-buip-estimation/results/max/lstr2_unrestricted/tables
  Aggregate Figures: /Users/lollo/Documents/Current_Projects/str-buip-estimation/results/max/lstr2_unrestricted/figures/aggregate
  Country Figures: /Users/lollo/Documents/Current_Projects/str-buip-estimation/results/max/lstr2_unrestricted/figures/country

Dataset Summary:
  Total observations: 6,721
  Countries: 14
  Date range: 1982-09-30 00:00:00 to 2025-07-31 00:00:00

Observations per country:
                  N      Start        End
country                                  
Australia       512 1982-09-30 2025-04-30
Brazil          514 1982-09-30 2025-06-30
Canada          514 1982-09-30 2025-06-30
Euro Area       318 1999-02-28 2025-07-31
Indonesia       468 1986-06-30 2025-05-31
Japan           479 1985-08-31 2025-06-30
Korea           513 1982-09-30 2025-05-31
Mexico          514 1982-09-30 2025-06-30


In [43]:
# Helpers: build candidate z's and prepare sample

import numpy as np
import pandas as pd

def _ensure_engineered_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df = df.sort_values('date')
    # Ensure r_s exists
    if 'r_s' not in df.columns:
        if 'rs' in df.columns:
            df['r_s'] = pd.to_numeric(df['rs'], errors='coerce')
        elif 'S' in df.columns:
            df['r_s'] = np.log(pd.to_numeric(df['S'], errors='coerce')).diff()
        elif 'S_log' in df.columns:
            df['r_s'] = pd.to_numeric(df['S_log'], errors='coerce').diff()
        else:
            df['r_s'] = np.nan
    # Ensure i_dom and i_for exist (prefer effective/monthly fallbacks)
    if 'i_dom' not in df.columns:
        for alt in ['i_dom_effective', 'i_dom_monthly', 'i_dom_m']:
            if alt in df.columns:
                df['i_dom'] = pd.to_numeric(df[alt], errors='coerce')
                break
    if 'i_for' not in df.columns:
        for alt in ['i_for_effective', 'i_for_monthly', 'i_for_m']:
            if alt in df.columns:
                df['i_for'] = pd.to_numeric(df[alt], errors='coerce')
                break
    # Forward-fill interest rates to reduce gaps
    if 'i_dom' in df.columns:
        df['i_dom'] = df['i_dom'].ffill()
    if 'i_for' in df.columns:
        df['i_for'] = df['i_for'].ffill()
    # Fundamental value q: use existing if present, else compute
    if 'q' not in df.columns or df['q'].isna().all():
        if all(col in df.columns for col in ['S','p_dom','p_for']):
            S = pd.to_numeric(df['S'], errors='coerce')
            p_dom = pd.to_numeric(df['p_dom'], errors='coerce')
            p_for = pd.to_numeric(df['p_for'], errors='coerce')
            df['q'] = np.log(S) - (np.log(p_dom) - np.log(p_for))
        elif all(col in df.columns for col in ['S','CPI_dom','CPI_for']):
            S = pd.to_numeric(df['S'], errors='coerce')
            CPI_dom = pd.to_numeric(df['CPI_dom'], errors='coerce')
            CPI_for = pd.to_numeric(df['CPI_for'], errors='coerce')
            df['q'] = np.log(S) - (np.log(CPI_dom) - np.log(CPI_for))
        else:
            df['q'] = np.nan
    return df

def prepare_unrestricted_sample(country_df: pd.DataFrame) -> pd.DataFrame:
    df = _ensure_engineered_columns(country_df)
    df = df.sort_values('date').copy().set_index('date')
    # Dep var: UIP-corrected return (excess returns)
    df['y'] = df['r_s'] - (df['i_for'] - df['i_dom'])
    df['rs_lag1'] = df['r_s'].shift(1)
    df['eta_lag1'] = df['q'].shift(1)
    # Coerce to numeric and drop impossible values
    for col in ['y','rs_lag1','eta_lag1','r_s','q','i_dom','i_for']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df


def build_lstr2_z_candidates(df_base: pd.DataFrame) -> dict:
    candidates = {}
    # Construct excess_returns explicitly for z candidates
    ex_ret = df_base['y']  # same as defined above
    for var_name, series in [('r_s', df_base['r_s']), ('q', df_base['q']), ('excess_returns', ex_ret)]:
        for k in range(1, 6):  # lags 1..5
            name = f'z__{var_name}_lag{k}'
            candidates[name] = series.shift(k)
    return candidates


def pick_quantile_grid(series: pd.Series) -> np.ndarray:
    s = pd.to_numeric(series, errors='coerce').dropna()
    if len(s) == 0:
        return np.array([0.0])
    qs = [0.10, 0.30, 0.50, 0.70, 0.90]
    return np.unique(np.array([float(np.quantile(s, q)) for q in qs], dtype=float))

In [44]:
# Sanity check: per-country sample sizes after engineering
min_required = 80  # adjust if needed
sizes = []
unique_countries = sorted(df_panel['country'].dropna().unique())
for country_label in unique_countries:
    sub = df_panel[df_panel['country'] == country_label].copy()
    base = prepare_unrestricted_sample(sub)
    essential_cols = [c for c in ['y','rs_lag1','eta_lag1'] if c in base.columns]
    if not essential_cols:
        n_eff = 0
    else:
        essential_ok = base[essential_cols].notna().all(axis=1)
        n_eff = int(essential_ok.sum())
    sizes.append({'Country': country_label, 'EffectiveRows': n_eff, 'MeetsThreshold': n_eff >= min_required})

sizes_df = pd.DataFrame(sizes).set_index('Country')
print('\n=== Engineered Sample Sizes (post-lagging) ===')
print(sizes_df)

# Optional: reduce threshold for countries with long histories but sparse early data
# min_required = 60


=== Engineered Sample Sizes (post-lagging) ===
                EffectiveRows  MeetsThreshold
Country                                      
Australia                 511            True
Brazil                    513            True
Canada                    513            True
Euro Area                 317            True
Indonesia                 467            True
Japan                     478            True
Korea                     512            True
Mexico                    513            True
New Zealand               482            True
Philippines               471            True
Switzerland               514            True
Thailand                  433            True
Türkiye                   469            True
United Kingdom            514            True


In [45]:
# Estimation loop per country with candidate selection
lstr2_results = {}  # country -> {'result': StrResult, 'z_name': str}

unique_countries = sorted(df_panel['country'].dropna().unique())
for country_label in unique_countries:
    print(f"\n=== {country_label} ===")
    try:
        sub = df_panel[df_panel['country'] == country_label].copy()
        base = prepare_unrestricted_sample(sub)
        # Build candidates
        cand_map = build_lstr2_z_candidates(base)
        best = None
        best_sse = np.inf
        best_start = None
        # Grids
        gamma_grid = np.logspace(np.log10(1.0), np.log10(50.0), num=6)
        for z_name, z_series in cand_map.items():
            df_cand = base.copy()
            df_cand[z_name] = z_series
            needed = [col for col in ['y','rs_lag1','eta_lag1', z_name] if col in df_cand.columns]
            df_cand = df_cand[needed].dropna()
            if len(df_cand) < 80:  # skip tiny samples
                continue
            model = STRModelNoRestrictionsLSTR2(df_cand, z_col=z_name)
            c_grid = pick_quantile_grid(df_cand[z_name])
            try:
                start = model.grid_search(gamma_grid=gamma_grid, c_grid=c_grid)
            except Exception:
                continue
            sse = start.get('sse', np.inf)
            if np.isfinite(sse) and sse < best_sse:
                best_sse = sse
                best = (z_name, df_cand)
                best_start = start
        if best is None:
            print('  No viable candidate found (insufficient data).')
            continue
        z_name, df_cand_best = best
        model_best = STRModelNoRestrictionsLSTR2(df_cand_best, z_col=z_name)
        res = model_best.fit(best_start, hac_lags=4)
        lstr2_results[country_label] = {'result': res, 'z_name': z_name}
        print(f"  ✓ Selected {z_name} | RMSE={res.rmse:.5f} | AIC={res.aic:.2f}")
    except Exception as e:
        print(f'  × Failed: {e}')


=== Australia ===
  ✓ Selected z__excess_returns_lag4 | RMSE=0.03187 | AIC=-3483.35

=== Brazil ===
  ✓ Selected z__excess_returns_lag1 | RMSE=0.07825 | AIC=-2596.11

=== Canada ===
  ✓ Selected z__excess_returns_lag2 | RMSE=0.01992 | AIC=-3991.88

=== Euro Area ===
  ✓ Selected z__excess_returns_lag1 | RMSE=0.07825 | AIC=-2596.11

=== Canada ===
  ✓ Selected z__excess_returns_lag2 | RMSE=0.01992 | AIC=-3991.88

=== Euro Area ===
  ✓ Selected z__excess_returns_lag3 | RMSE=0.02590 | AIC=-2283.69

=== Indonesia ===
  ✓ Selected z__excess_returns_lag3 | RMSE=0.02590 | AIC=-2283.69

=== Indonesia ===
  ✓ Selected z__r_s_lag5 | RMSE=0.05023 | AIC=-2751.73

=== Japan ===
  ✓ Selected z__r_s_lag2 | RMSE=0.02930 | AIC=-3349.93

=== Korea ===
  ✓ Selected z__r_s_lag5 | RMSE=0.05023 | AIC=-2751.73

=== Japan ===
  ✓ Selected z__r_s_lag2 | RMSE=0.02930 | AIC=-3349.93

=== Korea ===
  ✓ Selected z__excess_returns_lag5 | RMSE=0.02638 | AIC=-3675.35

=== Mexico ===
  ✓ Selected z__excess_returns_la

In [46]:
# Results summary table and save
rows = []
for c, obj in lstr2_results.items():
    res = obj['result']
    p = res.params
    rows.append({
        'Country': c,
        'z_name': obj['z_name'],
        'const0': p.get('const0', np.nan),
        'betac0': p.get('betac0', np.nan),
        'betaf0': p.get('betaf0', np.nan),
        'gamma': p.get('gamma', np.nan),
        'c1': p.get('c1', np.nan),
        'c2': p.get('c2', np.nan),
        'const1': p.get('const1', np.nan),
        'betac1': p.get('betac1', np.nan),
        'betaf1': p.get('betaf1', np.nan),
        'RMSE': res.rmse,
        'AIC': res.aic,
        'BIC': res.bic,
        'N': res.nobs
    })

df_summary = pd.DataFrame(rows).set_index('Country').sort_index()
print(df_summary.round(4))

csv_path = tables_dir / 'lstr2_unrestricted_results.csv'
tex_path = tables_dir / 'lstr2_unrestricted_results.tex'
df_summary.to_csv(csv_path)
with open(tex_path, 'w') as f:
    f.write(df_summary.to_latex(escape=False, caption='Unrestricted LSTR2 Estimates (Selected z and Parameters)', label='tab:lstr2_unrestricted'))
print(f'\nSaved: {csv_path.name}, {tex_path.name}')

# Optional: Aggregate G distribution across all countries
all_G = []
for c, obj in lstr2_results.items():
    all_G.append(obj['result'].G.dropna())
if all_G:
    G_all = pd.concat(all_G)
    fig, ax = plt.subplots(figsize=(7,4))
    ax.hist(G_all.values, bins=50, color='black', edgecolor='black', linewidth=0.8)
    ax.set_xlabel('Transition Function G(z)')
    ax.set_ylabel('Frequency')
    ax.set_title('LSTR2 Regime Distribution (Unrestricted)')
    plt.tight_layout()
    p_pdf = agg_fig_dir / 'lstr2_unrestricted_regime_distribution.pdf'
    p_png = agg_fig_dir / 'lstr2_unrestricted_regime_distribution.png'
    plt.savefig(p_pdf, bbox_inches='tight')
    plt.savefig(p_png, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved: {p_pdf.name}, {p_png.name}')
else:
    print('No G values to plot.')

                                z_name   const0   betac0  betaf0     gamma  \
Country                                                                      
Australia       z__excess_returns_lag4   0.0042   0.0692 -0.0269   28.9037   
Brazil          z__excess_returns_lag1 -22.9654  30.9228  1.2082    0.0165   
Canada          z__excess_returns_lag2  -0.0038   0.8859 -0.0133   50.0093   
Euro Area       z__excess_returns_lag3  -0.0022   0.1910 -0.0585    8.9248   
Indonesia                  z__r_s_lag5   0.0084  -0.0613 -0.0283    6.2332   
Japan                      z__r_s_lag2  -0.0013   0.2108 -0.0059  376.7791   
Korea           z__excess_returns_lag5   0.0058   0.7629  0.0128   67.7189   
Mexico          z__excess_returns_lag5   0.0019   0.0702 -0.0904   11.7459   
New Zealand                z__r_s_lag3  -0.0030   0.4017 -0.1287   52.6771   
Philippines                z__r_s_lag4   0.0072   0.0498 -0.0127   21.0537   
Switzerland     z__excess_returns_lag4   0.0074   0.2361  0.0725

In [48]:
# Detailed country plots (STR-style 3-panel stacked vertical layout)
import re
import matplotlib.pyplot as plt

def build_z_from_name(country_df, z_name):
    """Reconstruct the transition variable from its name."""
    m = re.match(r"z__([A-Za-z_]+)_lag(\d+)$", z_name)
    if not m:
        raise KeyError(f"Unrecognized z_name format: {z_name}")
    base, lag_str = m.group(1), m.group(2)
    k = int(lag_str)
    df = country_df.sort_index().copy()
    if base == 'r_s':
        series = df['r_s']
    elif base == 'q':
        series = df['q']
    elif base == 'excess_returns':
        series = df['r_s'] - (df['i_for'] - df['i_dom'])
    else:
        raise KeyError(f"Unsupported base in z_name: {base}")
    return series.shift(k)


def plot_lstr2_country_detailed(y: pd.Series, 
                                 transition_var: pd.Series,
                                 transition_func: pd.Series,
                                 c1: float,
                                 c2: float,
                                 country: str,
                                 save_path=None):
    """
    Creates a detailed 3-panel stacked vertical plot for LSTR2 model:
    - Top: Dependent variable (excess returns) over time
    - Middle: Transition variable with both thresholds c1, c2 (dashed lines)
    - Bottom: Transition function G(z) over time
    """
    # Create figure with 3 subplots, equal height ratios
    fig, axes = plt.subplots(3, 1, figsize=(12, 6), 
                            gridspec_kw={'height_ratios': [1, 1, 1]}, dpi=300)
    fig.patch.set_facecolor('white')
    
    # Ensure all series are aligned and have same index
    common_idx = y.index.intersection(transition_var.index).intersection(transition_func.index)
    y = y.loc[common_idx]
    transition_var = transition_var.loc[common_idx]
    transition_func = transition_func.loc[common_idx]
    
    # ==========================================
    # TOP PANEL: Dependent Variable (Excess Returns)
    # ==========================================
    ax_top = axes[0]
    ax_top.plot(y.index, y.values, color='black', linewidth=0.7)
    ax_top.spines['top'].set_visible(True)
    ax_top.spines['right'].set_visible(True)
    ax_top.spines['bottom'].set_visible(True)
    ax_top.spines['left'].set_visible(True)
    ax_top.set_ylabel('')
    ax_top.set_xlabel('')
    ax_top.grid(False)
    plt.setp(ax_top.xaxis.get_majorticklabels(), rotation=0, ha='center')
    
    # ==========================================
    # MIDDLE PANEL: Transition Variable + Both Thresholds
    # ==========================================
    ax_mid = axes[1]
    ax_mid.plot(transition_var.index, transition_var.values, 
                color='black', linewidth=0.7)
    # Plot both thresholds with dashed lines
    if c1 is not None:
        ax_mid.axhline(c1, color='black', linestyle='--', linewidth=0.7)
    if c2 is not None:
        ax_mid.axhline(c2, color='black', linestyle='--', linewidth=0.7)
    ax_mid.spines['top'].set_visible(True)
    ax_mid.spines['right'].set_visible(True)
    ax_mid.spines['bottom'].set_visible(True)
    ax_mid.spines['left'].set_visible(True)
    ax_mid.set_ylabel('')
    ax_mid.set_xlabel('')
    ax_mid.grid(False)
    plt.setp(ax_mid.xaxis.get_majorticklabels(), rotation=0, ha='center')
    
    # ==========================================
    # BOTTOM PANEL: Transition Function G(z)
    # ==========================================
    ax_bot = axes[2]
    ax_bot.plot(transition_func.index, transition_func.values, 
                color='black', linewidth=0.7)
    ax_bot.set_ylim(0.0, 1.0)
    ax_bot.spines['top'].set_visible(True)
    ax_bot.spines['right'].set_visible(True)
    ax_bot.spines['bottom'].set_visible(True)
    ax_bot.spines['left'].set_visible(True)
    ax_bot.set_ylabel('')
    ax_bot.set_xlabel('')
    ax_bot.grid(False)
    plt.setp(ax_bot.xaxis.get_majorticklabels(), rotation=0, ha='center')
    
    plt.tight_layout()
    
    if save_path:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
        fig.savefig(save_path.with_suffix('.png'), dpi=300, bbox_inches='tight', facecolor='white')
        plt.close(fig)
    
    return fig


print("\nLSTR2 detailed plots (3-panel stacked style):")
if lstr2_results:
    for country_name, result_dict in lstr2_results.items():
        try:
            str_result = result_dict['result']
            z_name = result_dict['z_name']
            
            # Get country data
            country_df = df_panel[df_panel['country'] == country_name].copy().set_index('date').sort_index()
            
            # Build transition variable z
            z_series = build_z_from_name(country_df, z_name)
            
            # Dependent variable y (excess returns)
            y = country_df['r_s'] - (country_df['i_for'] - country_df['i_dom'])
            
            # Transition function G from estimation result
            G = str_result.G
            
            # Thresholds
            c1 = str_result.params.get('c1', None)
            c2 = str_result.params.get('c2', None)
            
            # Plot
            save_path = country_fig_dir / f"{country_name.replace(' ', '_')}_str_detailed.pdf"
            plot_lstr2_country_detailed(
                y=y,
                transition_var=z_series,
                transition_func=G,
                c1=c1,
                c2=c2,
                country=country_name,
                save_path=save_path
            )
            print(f"  ✓ {country_name}")
        except Exception as e:
            print(f"  ✗ {country_name}: {e}")
else:
    print('No LSTR2 results available.')


LSTR2 detailed plots (3-panel stacked style):
  ✓ Australia
  ✓ Australia
  ✓ Brazil
  ✓ Brazil
  ✓ Canada
  ✓ Canada
  ✓ Euro Area
  ✓ Euro Area
  ✓ Indonesia
  ✓ Indonesia
  ✓ Japan
  ✓ Japan
  ✓ Korea
  ✓ Korea
  ✓ Mexico
  ✓ Mexico
  ✓ New Zealand
  ✓ New Zealand
  ✓ Philippines
  ✓ Philippines
  ✓ Switzerland
  ✓ Switzerland
  ✓ Thailand
  ✓ Thailand
  ✓ Türkiye
  ✓ Türkiye
  ✓ United Kingdom
  ✓ United Kingdom
